# Dataset analysis

Post-split composition for `COLLECTION`: per-class counts, loss weights, STFT pilots, clip levels.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os

import numpy as np
import matplotlib.pyplot as plt
import pyrootutils
import soundfile as sf
from tqdm.auto import tqdm

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

## Parameters

`COLLECTION` is the on-disk dataset folder. STFT parameters match `building/utils/constants.py`.

In [ ]:
COLLECTION = "task_s3_49_2_r50"
NON_TARGET_NAME = "non_target"
MAX_CLIPS_PER_GROUP = 400  # cap per class for STFT averaging
RNG = np.random.default_rng(42)

from building.polychirp_training.training import load_dataset_catalog

catalog = load_dataset_catalog(COLLECTION)
dataset_root = ROOT / "datasets" / COLLECTION
TARGET_SPECIES = [n for n in catalog.class_names if n != NON_TARGET_NAME]
print(f"{COLLECTION}  |  {len(TARGET_SPECIES)} targets + {NON_TARGET_NAME}")

## Per-class sample counts + loss weights

Catalog split counts and the train-split inverse-frequency loss weights.

In [ ]:
import pandas as pd

from building.polychirp_training.training import inverse_freq_weights

non_target_idx = catalog.class_names.index(NON_TARGET_NAME)
target_idxs = [i for i, n in enumerate(catalog.class_names) if n != NON_TARGET_NAME]
ordered = target_idxs + [non_target_idx]  # targets first, non_target last

# Inverse-frequency weights from train-split counts of the merged class space.
train_counts = {lbl: catalog.entries[g].train.count for lbl, g in enumerate(ordered)}
weights = inverse_freq_weights(train_counts, len(ordered), sum(train_counts.values()))
weights_by_name = {catalog.class_names[g]: weights[lbl] for lbl, g in enumerate(ordered)}

rows = [
    (
        e.name,
        e.train.count,
        e.val.count,
        e.test.count,
        e.train.count + e.val.count + e.test.count,
        weights_by_name.get(e.name),
    )
    for e in catalog.entries
]
counts = pd.DataFrame(
    rows, columns=["class", "train", "val", "test", "total", "weight"]
).set_index("class")
counts.loc["TOTAL"] = [
    counts["train"].sum(),
    counts["val"].sum(),
    counts["test"].sum(),
    counts["total"].sum(),
    None,
]

(
    counts.style
    .background_gradient(cmap="Blues", subset=(counts.index[:-1], ["train", "val", "test"]))
    .format(
        {"train": "{:,.0f}", "val": "{:,.0f}", "test": "{:,.0f}",
         "total": "{:,.0f}", "weight": "{:.3f}"},
        na_rep="-",
    )
    .set_caption(f"Sample counts + per-class loss weights  ({COLLECTION})")
    .set_table_styles([
        {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "110%"), ("padding", "6px 0")]},
        {"selector": "th", "props": [("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
    ])
)

## Spectrogram helpers

Read every class from `datasets/<COLLECTION>/{training,validation,testing}/<class>/`.

In [ ]:
from building.utils.constants import (
    SAMPLE_RATE,
    TARGET_AUDIO_LEN,
    FRAME_LENGTH,
    FRAME_STEP,
)

SPLITS = ("training", "validation", "testing")


def class_clips(name: str) -> list[Path]:
    """Every assembled clip for one class across the three splits."""
    out: list[Path] = []
    for split in SPLITS:
        d = dataset_root / split / name
        if d.exists():
            out.extend(sorted(d.glob("*.wav")))
    return out


def load_clip(path: Path) -> np.ndarray:
    x, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if x.ndim > 1:
        x = x.mean(axis=-1)
    if sr != SAMPLE_RATE:
        n_new = int(round(len(x) * SAMPLE_RATE / sr))
        x = np.interp(np.linspace(0, len(x), n_new, endpoint=False), np.arange(len(x)), x).astype(np.float32)
    if len(x) >= TARGET_AUDIO_LEN:
        return x[:TARGET_AUDIO_LEN]
    pad = np.zeros(TARGET_AUDIO_LEN - len(x), dtype=np.float32)
    return np.concatenate([x, pad])


window = np.hanning(FRAME_LENGTH).astype(np.float32)
n_frames = 1 + (TARGET_AUDIO_LEN - FRAME_LENGTH) // FRAME_STEP
n_freqs = FRAME_LENGTH // 2 + 1
stft_freqs = np.fft.rfftfreq(FRAME_LENGTH, d=1.0 / SAMPLE_RATE)
stft_times = (np.arange(n_frames) * FRAME_STEP + FRAME_LENGTH / 2) / SAMPLE_RATE


def stft_mag(x: np.ndarray) -> np.ndarray:
    """Magnitude STFT, shape (n_freqs, n_frames)."""
    frames = np.lib.stride_tricks.sliding_window_view(x, FRAME_LENGTH)[::FRAME_STEP]
    frames = frames * window
    return np.abs(np.fft.rfft(frames, axis=-1)).T.astype(np.float32)


def to_db(mag: np.ndarray, ref: float | None = None) -> np.ndarray:
    ref = float(mag.max()) if ref is None else ref
    return 20.0 * np.log10(np.maximum(mag, 1e-10) / max(ref, 1e-10))


def avg_stft(files: list[Path], cap: int | None = None) -> tuple[np.ndarray, int]:
    if cap is not None and len(files) > cap:
        idx = RNG.choice(len(files), size=cap, replace=False)
        files = [files[i] for i in idx]
    acc = np.zeros((n_freqs, n_frames), dtype=np.float64)
    n = 0
    for f in tqdm(files, desc=f"avg-stft ({len(files)})", leave=False):
        try:
            acc += stft_mag(load_clip(f))
            n += 1
        except Exception:
            pass
    return (acc / max(n, 1)).astype(np.float32), n


def safe(name: str) -> str:
    """Class name as a valid npz key (no spaces)."""
    return name.replace(" ", "_")

## Average STFT: targets vs the non_target pool

Average magnitude STFT over target clips vs the pooled non_target clips. Cached to `analysis_cache.npz`.

In [ ]:
cache_path = dataset_root / "analysis_cache.npz"


def gather_files(names: list[str]) -> list[Path]:
    out: list[Path] = []
    for nm in names:
        out.extend(class_clips(nm))
    return out


def compute_cache() -> dict:
    payload: dict[str, np.ndarray] = {}
    target_files = gather_files(TARGET_SPECIES)
    other_files = class_clips(NON_TARGET_NAME)
    print(f"target pool    : {len(target_files)} clips")
    print(f"non_target pool: {len(other_files)} clips")
    cap = MAX_CLIPS_PER_GROUP * max(1, len(TARGET_SPECIES))
    avg_t, n_t = avg_stft(target_files, cap=cap)
    avg_o, n_o = avg_stft(other_files, cap=cap)
    payload["avg_target"] = avg_t
    payload["avg_other"] = avg_o
    payload["n_target"] = np.array(n_t)
    payload["n_other"] = np.array(n_o)
    for sp in TARGET_SPECIES:
        files = class_clips(sp)
        if not files:
            continue
        avg_sp, n_sp = avg_stft(files, cap=MAX_CLIPS_PER_GROUP)
        payload[f"species__{safe(sp)}"] = avg_sp
        payload[f"count__{safe(sp)}"] = np.array(n_sp)
    np.savez(cache_path, **payload)
    return payload


if cache_path.exists():
    cache = dict(np.load(cache_path, allow_pickle=False))
    print(f"loaded cache: {cache_path}")
else:
    cache = compute_cache()
    print(f"wrote cache: {cache_path}")

print(f"  n_target = {int(cache['n_target'])}")
print(f"  n_other  = {int(cache['n_other'])}")

In [ ]:
avg_t = cache["avg_target"]
avg_o = cache["avg_other"]
ref = float(max(avg_t.max(), avg_o.max()))

fig, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
for ax, mag, label, n in [
    (axes[0], avg_t, "Average STFT: target species", int(cache["n_target"])),
    (axes[1], avg_o, "Average STFT: pooled non_target", int(cache["n_other"])),
]:
    im = ax.imshow(
        to_db(mag, ref=ref),
        origin="lower",
        aspect="auto",
        extent=[0, stft_times[-1], stft_freqs[0], stft_freqs[-1]],
        vmin=-70,
        vmax=0,
        cmap="magma",
    )
    ax.set_yscale("symlog", linthresh=200)
    ax.set_yticks([0, 500, 1000, 4000, 8000])
    ax.set_yticklabels(["0", "500", "1k", "4k", "8k"])
    ax.set_ylabel("Hz")
    ax.set_title(f"{label}  (n={n})", fontsize=10)
axes[-1].set_xlabel("Time (s)")
fig.colorbar(im, ax=axes.ravel().tolist(), label="dB (relative to joint peak)")
plt.show()

## Per-species average STFT

In [ ]:
species = [sp for sp in TARGET_SPECIES if f"species__{safe(sp)}" in cache]
n = len(species)
cols = 5
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3.0 * cols, 2.4 * rows), squeeze=False, sharex=True, sharey=True)
ref = float(max(cache[f"species__{safe(sp)}"].max() for sp in species))
for ax, sp in zip(axes.ravel(), species):
    mag = cache[f"species__{safe(sp)}"]
    ax.imshow(
        to_db(mag, ref=ref),
        origin="lower",
        aspect="auto",
        extent=[0, stft_times[-1], stft_freqs[0], stft_freqs[-1]],
        vmin=-70,
        vmax=0,
        cmap="magma",
    )
    ax.set_yscale("symlog", linthresh=200)
    ax.set_yticks([0, 500, 1000, 4000, 8000])
    ax.set_yticklabels(["0", "500", "1k", "4k", "8k"])
    ax.set_title(f"{sp}  (n={int(cache[f'count__{safe(sp)}'])})", fontsize=8)
for ax in axes.ravel()[n:]:
    ax.axis("off")
fig.supxlabel("Time (s)")
fig.supylabel("Frequency (Hz)")
fig.suptitle("Per-species average STFT (dB, common reference)")
plt.tight_layout()
plt.show()

## Per-species difference vs the non_target pool

Per-species STFT minus non_target STFT in dB: red louder than the average bird, blue quieter. Each species dB-normalised to its own peak.

In [ ]:
avg_o = cache["avg_other"]
ref_o = float(avg_o.max())
other_db = to_db(avg_o, ref=ref_o)

n = len(species)
cols = 5
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3.0 * cols, 2.4 * rows), squeeze=False, sharex=True, sharey=True)
diffs = {}
for sp in species:
    mag = cache[f"species__{safe(sp)}"]
    sp_db = to_db(mag, ref=float(mag.max()))
    diffs[sp] = sp_db - other_db
vmax = max(np.percentile(np.abs(d), 98) for d in diffs.values())
for ax, sp in zip(axes.ravel(), species):
    im = ax.imshow(
        diffs[sp],
        origin="lower",
        aspect="auto",
        extent=[0, stft_times[-1], stft_freqs[0], stft_freqs[-1]],
        vmin=-vmax,
        vmax=vmax,
        cmap="RdBu_r",
    )
    ax.set_yscale("symlog", linthresh=200)
    ax.set_yticks([0, 500, 1000, 4000, 8000])
    ax.set_yticklabels(["0", "500", "1k", "4k", "8k"])
    ax.set_title(sp, fontsize=8)
for ax in axes.ravel()[n:]:
    ax.axis("off")
fig.supxlabel("Time (s)")
fig.supylabel("Frequency (Hz)")
fig.suptitle("Per-species STFT minus non_target STFT  (dB, +/-{:.0f} dB)".format(vmax))
fig.colorbar(im, ax=axes.ravel().tolist(), label="dB difference")
plt.show()

## Marginal frequency profile

Time-averaged spectrum per species vs the non_target pool.

In [ ]:
avg_o = cache["avg_other"]
other_profile = to_db(avg_o.mean(axis=1), ref=float(avg_o.mean(axis=1).max()))

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(9, 7.5), sharex=True)

# Absolute spectrum, each curve dB-normalised to its own peak.
for sp in species:
    mag = cache[f"species__{safe(sp)}"]
    p = mag.mean(axis=1)
    p_db = to_db(p, ref=float(p.max()))
    ax_top.plot(stft_freqs, p_db, lw=1.1, label=sp)
ax_top.plot(stft_freqs, other_profile, color="k", lw=2, ls="--", label="non_target pool")
ax_top.set_ylabel("dB (rel. own peak)")
ax_top.set_ylim(-60, 2)
ax_top.legend(fontsize=7, ncol=2, loc="lower center")
ax_top.set_title("Time-averaged spectrum: absolute (each curve rel. own peak)")

# Difference vs the non_target pool: positive louder, negative quieter.
for sp in species:
    mag = cache[f"species__{safe(sp)}"]
    p = mag.mean(axis=1)
    p_db = to_db(p, ref=float(p.max()))
    ax_bot.plot(stft_freqs, p_db - other_profile, lw=1.1, label=sp)
ax_bot.axhline(0, color="k", lw=1, ls="--", alpha=0.5)
ax_bot.set_xlabel("Frequency (Hz)")
ax_bot.set_ylabel("dB diff vs non_target")
ax_bot.set_title("Time-averaged spectrum: difference vs the non_target pool")

for ax in (ax_top, ax_bot):
    ax.set_xscale("symlog", linthresh=200)
    ax.set_xticks([0, 500, 1000, 2000, 4000, 8000])
    ax.set_xticklabels(["0", "500", "1k", "2k", "4k", "8k"])
    ax.set_xlim(0, stft_freqs[-1])  # symlog pads into negatives by default
plt.tight_layout()
plt.show()

## Per-class RMS distribution

Clip levels per class, 2 dB bins from -70 to 0 dBFS.

In [ ]:
def rms_dbfs(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float32)
    return float(20.0 * np.log10(max(np.sqrt(np.mean(x**2)), 1e-10)))


def scan(files: list[Path]) -> np.ndarray:
    out: list[float] = []
    for f in tqdm(files, desc=f"scan ({len(files)})", leave=False):
        try:
            x, _ = sf.read(str(f), dtype="float32", always_2d=False)
            if x.ndim > 1:
                x = x.mean(axis=-1)
            out.append(rms_dbfs(x))
        except Exception:
            pass
    return np.array(out)


groups = [(sp, scan(class_clips(sp))) for sp in TARGET_SPECIES]
groups.append((NON_TARGET_NAME, scan(class_clips(NON_TARGET_NAME))))

print(f"{'class':<32s} {'n':>5s}  {'q05':>6s} {'q50':>6s} {'q95':>6s}  {'min':>6s} {'max':>6s}")
for label, a in groups:
    if len(a) == 0:
        continue
    print(
        f"{label:<32s} {len(a):>5d}  "
        f"{np.quantile(a, 0.05):>+6.1f} {np.median(a):>+6.1f} {np.quantile(a, 0.95):>+6.1f}  "
        f"{a.min():>+6.1f} {a.max():>+6.1f}"
    )

bins = np.arange(-70, 1, 2)
n = len(groups)
fig, axes = plt.subplots(n, 1, figsize=(10, max(2.0, 1.6 * n)), sharex=True)
if n == 1:
    axes = [axes]
for ax, (label, a) in zip(axes, groups):
    if len(a):
        ax.hist(a, bins=bins, color="steelblue", edgecolor="black", alpha=0.85)
        ax.axvline(np.median(a), color="red", lw=1, ls="--", label=f"median {np.median(a):+.1f}")
        ax.legend(loc="upper left", fontsize=8, frameon=False)
    ax.set_ylabel(label, rotation=0, ha="right", va="center")
axes[-1].set_xlabel("RMS (dBFS)")
fig.suptitle("Per-class RMS distribution (assembled dataset)")
plt.tight_layout()
plt.show()